# gufe framejs visualizations

Interactive `.view()` / bare-cell rendering for gufe objects, through
[framejs.io](https://framejs.io). Runs on **gufe** alone — no `openfe` needed.

**Run the setup cell, then run any view cell below.** The views come first so you
can iterate on a frame (`gufe/visualization/viz_assets/<frame>/code.js`), restart
the kernel, and re-run just the section you're working on. Everything
non-visual — how the machinery works, the frame registry, CLI URLs, fallback
behaviour — is under [Reference](#reference) at the bottom.

> Started with `just dev` → open me at **http://localhost:8888/lab**.

## Setup

One cell that builds one of everything from gufe's own test data. Every view
below depends on it.

In [ ]:
import warnings
from pathlib import Path

import gufe
from gufe.tests.test_protocol import DummyProtocol

warnings.filterwarnings("ignore", message=".*hydrogen atoms.*")

data = Path(gufe.__file__).parent / "tests" / "data"

# the demo ligand network, and one edge off it -> a ligand pair + atom mapping
net = gufe.LigandNetwork.from_graphml((data / "ligand_network.graphml").read_text())
edge = sorted(net.edges, key=lambda e: str(e.key))[0]
ligand_A, ligand_B = edge.componentA, edge.componentB

protein = gufe.ProteinComponent.from_pdb_file(str(data / "181l.pdb"), name="T4 lysozyme")
solvent = gufe.SolventComponent()

# two chemical systems that differ only in which ligand is bound
stateA = gufe.ChemicalSystem({"ligand": ligand_A, "protein": protein, "solvent": solvent}, name="complex A")
stateB = gufe.ChemicalSystem({"ligand": ligand_B, "protein": protein, "solvent": solvent}, name="complex B")

protocol = DummyProtocol(settings=DummyProtocol.default_settings())
transformation = gufe.Transformation(
    stateA=stateA, stateB=stateB, protocol=protocol, mapping=edge, name="A->B"
)
alchemical_net = gufe.AlchemicalNetwork(edges=[transformation], name="demo campaign")

# the graphml test ligands are unnamed, so fall back to their gufe key
label = lambda c: c.name or str(c.key)

print(f"ligands   {label(ligand_A)} -> {label(ligand_B)}")
print(f"mapping   {len(edge.componentA_to_componentB)} atoms mapped")
print(f"network   {len(net.nodes)} ligands, {len(net.edges)} edges")
print(f"campaign  {len(alchemical_net.nodes)} nodes, {len(alchemical_net.edges)} edges")

---

# The views

One section per frame, smallest object first. A bare object on the last line
renders itself; `.view()` is the same widget with `width` / `height` overrides.

### `SmallMoleculeComponent`

2D depiction (RDKit-js) beside an interactive 3D view of the same conformer, with
SMILES / charge / atom counts underneath.

In [ ]:
ligand_A

### `SolventComponent`

No coordinates to show — a solvent component is a *specification*, so it renders
as a compact settings card with an illustrative box of ions.

In [ ]:
solvent

### `ProteinComponent`

3Dmol.js with representation and colour-scheme switchers. Waters are hidden by
default; the toolbar carries chain / residue / atom counts. This one works for
`SolvatedPDBComponent` and `ProteinMembraneComponent` too — they inherit the viz
through the MRO.

In [ ]:
protein.view(height="600px")

### `LigandAtomMapping`

The atom correspondence between two ligands — the same viewer that drives the
right-hand pane of the `LigandNetwork` viz, standalone. Switch between
**plain / colored / lines / overlay / 2d**: core atoms grey, unique atoms red (A)
and green (B).

In [ ]:
edge

### `ChemicalSystem`

Master/detail over the system's labelled components: pick a component on the left,
see it rendered on the right according to its type.

In [ ]:
stateA.view(height="600px")

### `Transformation`

The point of a transformation is *what changes* between its two states, so the
frame leads with a component diff (unchanged / changed / added / removed), and
puts the atom mapping underneath.

In [ ]:
transformation.view(height="700px")

### `LigandNetwork`

Radial graph of the ligands with the per-edge atom mapping on the right. Also the
bare-cell demo: `net` is the last line of the cell, no `.view()` call.

In [ ]:
net  # bare-cell → interactive framejs radial graph

Same widget, called explicitly with a size override.

In [ ]:
net.view(height="600px")

### `AlchemicalNetwork`

A d3 force graph of `ChemicalSystem` nodes and `Transformation` edges. Nodes are
coloured by their component *signature*, so the complex leg and the solvent leg of
a campaign separate visually. Click a node or edge for details; drag to rearrange.

In [ ]:
alchemical_net.view(height="600px")

---

# Reference

<a id="reference"></a>

Nothing below draws anything — it's the machinery, the registry, and the
non-visual checks. Skip it while iterating on a frame.

## How a view reaches the browser

Each object pushes its own data to a canonical published frame over the widget
comm channel, so there's no URL size limit in the notebook.

- `.view(width=…, height=…)` returns the widget explicitly.
- A bare object on the last line of a cell renders through `_repr_mimebundle_`.

Both paths build the same widget; the bare-cell one just uses IPython's display
hook instead of an explicit call.

## The frame registry

Each viewable gufe type maps to one canonical framejs frame on disk under
`gufe/visualization/viz_assets/<frame>/`, named after the class it renders:

| gufe class | frame |
| --- | --- |
| `LigandNetwork` | `ligand_network` |
| `AlchemicalNetwork` | `alchemical_network` |
| `Transformation` / `NonTransformation` | `transformation` |
| `ChemicalSystem` | `chemical_system` |
| `LigandAtomMapping` | `ligand_atom_mapping` |
| `SmallMoleculeComponent` | `small_molecule_component` |
| `ProteinComponent` (+ solvated/membrane subclasses) | `protein_component` |
| `SolventComponent` | `solvent_component` |

Opting a class in is one mixin — `FramejsViewable` — plus a `VizRef` and a payload
builder in `gufe/visualization/framejs.py`. Lookup walks the MRO, so subclasses
inherit their parent's viz for free.

`has_local()` below is the one to check while iterating: `True` means the view is
being served from your working copy of that frame's `code.js`.

In [ ]:
from gufe.visualization import framejs

for cls_name, viz in sorted(framejs.VIZ_REGISTRY.items()):
    print(f"{cls_name:24} -> viz_assets/{viz.frame:26} on disk: {viz.has_local()}")

### Shareable CLI URLs

The same vizzes, without a live kernel. `build_cli_url` takes the frame's
self-contained URL and appends this object's data as `#?inputs=<base64>` — that's
what the CLI hands to `webbrowser.open()`. It works for every registered type.

The URLs are long enough to be useless as notebook output, so only lengths and a
prefix are printed here. Note the sizes — a protein inlines megabytes of PDB into the URL.

In [ ]:
from gufe.visualization.framejs import build_cli_url

for obj in (net, ligand_A, edge, solvent, stateA, transformation, alchemical_net):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        url = build_cli_url(obj)
    print(f"{type(obj).__name__:24} {len(url):>9,} chars   {url[:60]}...")


### Graceful degradation

Nothing here is load-bearing. Without the `viz` extra installed, `.view()` falls
back to the object's legacy RDKit / py3Dmol renderer (or `None`), and
`_repr_mimebundle_` returns `None` so a bare cell just prints the normal `repr`.

In [ ]:
from gufe.visualization.framejs import repr_mimebundle

# an object with no registered viz must never break bare-cell display
print("unregistered ->", repr_mimebundle(object()))

# the legacy 2D mapping drawing is still right there
edge._ipython_display_()

## Real OpenFE demo inputs (optional)

`just dev` also clones **openfe-demo** locally (git-ignored) at
`/workspace/openfe-demo`. Its `src/inputs/` has real ligand/protein files you can
load into gufe objects. The full `openfe_demo.ipynb` there needs the heavy
`openfe` env, but the input data is handy for exercising the gufe vizzes.

In [ ]:
inputs = Path("/workspace/openfe-demo/src/inputs")
sorted(p.name for p in inputs.glob("*")) if inputs.exists() else "run `just dev` to clone openfe-demo"